In [3]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.dummy import DummyRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", None)

print("Libraries imported successfully.")

Libraries imported successfully.


Configure project directories

In [4]:
project_dir = Path(
    r"C:\Users\U S E R\Desktop\ML Assigment\project"
)

processed_dir = project_dir / "data" / "processed"

results_dir = processed_dir / "model_comparisons"

models_dir = project_dir / "models"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Results directory:", results_dir)
print("Models directory:", models_dir)

Results directory: C:\Users\U S E R\Desktop\ML Assigment\project\data\processed\model_comparisons
Models directory: C:\Users\U S E R\Desktop\ML Assigment\project\models


Load the processed training dataset

In [5]:
train_df = pd.read_csv(
    processed_dir / "math_train_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

assert X_train.notna().all().all()
assert y_train.notna().all()

print("Training samples:", X_train.shape[0])
print("Number of features:", X_train.shape[1])
print("Target variable: G3")

display(train_df.head())

Training samples: 316
Number of features: 58
Target variable: G3


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,num__Walc,num__health,num__G1,num__G2,absences_log__absences,cat__school_GP,cat__school_MS,cat__sex_F,cat__sex_M,cat__address_R,cat__address_U,cat__famsize_GT3,cat__famsize_LE3,cat__Pstatus_A,cat__Pstatus_T,cat__Mjob_at_home,cat__Mjob_health,cat__Mjob_other,cat__Mjob_services,cat__Mjob_teacher,cat__Fjob_at_home,cat__Fjob_health,cat__Fjob_other,cat__Fjob_services,cat__Fjob_teacher,cat__reason_course,cat__reason_home,cat__reason_other,cat__reason_reputation,cat__guardian_father,cat__guardian_mother,cat__guardian_other,cat__schoolsup_no,cat__schoolsup_yes,cat__famsup_no,cat__famsup_yes,cat__paid_no,cat__paid_yes,cat__activities_no,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-0.5,0.2,0.50,-0.232487,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
1,-0.5,-0.5,0.0,1.0,-1.0,0.0,1.0,0.0,0.0,0.0,-0.5,-0.5,0.4,0.75,-0.732487,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,14
2,-0.5,-1.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,-0.5,-0.5,-0.6,-1.00,-0.732487,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0
3,-0.5,0.5,0.0,0.0,1.0,0.0,-1.0,1.0,0.5,1.0,1.0,0.0,-0.2,-0.50,-0.232487,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,9
4,-0.5,-0.5,-2.0,1.0,-1.0,0.0,-1.0,0.0,-0.5,0.0,0.5,-0.5,-0.6,-0.50,-0.732487,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,8


Initialize the four best models

In [6]:
best_models = {

    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(
        alpha=10.0
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42
    )
}

model_experiments = {
    "Linear Regression": "LR-1 All Features",
    "Ridge Regression": "Ridge-4",
    "Random Forest": "RF-4",
    "Gradient Boosting": "GB-5"
}

model_filenames = {
    "Linear Regression": "best_linear_regression.joblib",
    "Ridge Regression": "best_ridge_regression.joblib",
    "Random Forest": "best_random_forest.joblib",
    "Gradient Boosting": "best_gradient_boosting.joblib"
}

for name, model in best_models.items():
    print(f"\n{name}")
    print(model)


Linear Regression
LinearRegression()

Ridge Regression
Ridge(alpha=10.0)

Random Forest
RandomForestRegressor(max_depth=15, min_samples_leaf=2, n_estimators=300,
                      n_jobs=-1, random_state=42)

Gradient Boosting
GradientBoostingRegressor(max_depth=4, min_samples_leaf=2, n_estimators=300,
                          random_state=42)


Configure cross-validation

In [7]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("Cross-validation: 5-Fold KFold")
print("Shuffle: True")
print("Random State: 42")

Cross-validation: 5-Fold KFold
Shuffle: True
Random State: 42


Train and evaluate all four models

In [9]:
model_results = []
fold_results = []
oof_predictions = {}

actual = y_train.to_numpy()

actual_binary = (actual >= 10).astype(int)

for model_name, model in best_models.items():

    print("=" * 65)
    print("Training:", model_name)
    print("=" * 65)

    # 1. Cross-validation
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise",
        n_jobs=1
    )

    # 2. Out-of-fold predictions
    predictions = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv,
        n_jobs=1
    )

    oof_predictions[model_name] = predictions

    # 3. Convert numerical predictions to integer grades
    rounded_predictions = np.clip(
        np.rint(predictions),
        0,
        20
    ).astype(int)

    # 4. Convert numerical grades to pass/fail
    predicted_binary = (
        predictions >= 10
    ).astype(int)

    # 5. Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    # 6. Training and validation RMSE
    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    # 7. Store overall metrics
    model_results.append({

        "Model": model_name,
        "Experiment": model_experiments[model_name],

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

    # 8. Store individual fold metrics
    for fold in range(5):

        fold_results.append({
            "Model": model_name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train MSE": -scores["train_MSE"][fold],
            "Validation MSE": -scores["test_MSE"][fold],

            "Train RMSE": np.sqrt(
                -scores["train_MSE"][fold]
            ),

            "Validation RMSE": np.sqrt(
                -scores["test_MSE"][fold]
            ),

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

    print(f"Validation RMSE: {val_rmse:.4f}")
    print(f"Validation MAE : {-scores['test_MAE'].mean():.4f}")
    print("Completed successfully.\n")

results_df = pd.DataFrame(model_results)
fold_df = pd.DataFrame(fold_results)

print("All four models trained and evaluated successfully.")

Training: Linear Regression
Validation RMSE: 1.8829
Validation MAE : 1.3724
Completed successfully.

Training: Ridge Regression
Validation RMSE: 1.8322
Validation MAE : 1.2920
Completed successfully.

Training: Random Forest
Validation RMSE: 1.4733
Validation MAE : 0.9324
Completed successfully.

Training: Gradient Boosting
Validation RMSE: 1.4427
Validation MAE : 0.9385
Completed successfully.

All four models trained and evaluated successfully.


Rank the models by validation RMSE

In [10]:
results_df = results_df.sort_values(
    by=["Validation RMSE", "Validation MAE"],
    ascending=[True, True]
).reset_index(drop=True)

results_df.insert(
    0,
    "Rank",
    np.arange(1, len(results_df) + 1)
)

print("=" * 70)
print("FINAL MODEL RANKING")
print("=" * 70)

display(
    results_df[
        [
            "Rank",
            "Model",
            "Experiment",
            "Validation RMSE",
            "Validation MAE",
            "Validation R2",
            "RMSE Gap"
        ]
    ].round(4)
)

FINAL MODEL RANKING


,Rank,Model,Experiment,Validation RMSE,Validation MAE,Validation R2,RMSE Gap
0,1,Gradient Boosting,GB-5,1.4427,0.9385,0.8950,1.4017
1,2,Random Forest,RF-4,1.4733,0.9324,0.8933,0.7417
2,3,Ridge Regression,Ridge-4,1.8322,1.2920,0.8328,0.2626
3,4,Linear Regression,LR-1 All Features,1.8829,1.3724,0.8239,0.3337


Compare grade prediction accuracy

In [11]:
grade_columns = [
    "Rank",
    "Model",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

grade_comparison = results_df[grade_columns]

print("=" * 70)
print("GRADE PREDICTION ACCURACY COMPARISON")
print("=" * 70)

display(grade_comparison.round(2))

GRADE PREDICTION ACCURACY COMPARISON


,Rank,Model,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,1,Gradient Boosting,43.99,68.99,89.56
1,2,Random Forest,48.10,70.57,89.56
2,3,Ridge Regression,30.38,53.80,82.91
3,4,Linear Regression,30.38,51.90,78.80


Compare pass/fail classification

In [12]:
classification_columns = [
    "Rank",
    "Model",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

classification_comparison = results_df[
    classification_columns
]

print("=" * 70)
print("PASS/FAIL CLASSIFICATION COMPARISON")
print("=" * 70)

display(classification_comparison.round(4))

PASS/FAIL CLASSIFICATION COMPARISON


,Rank,Model,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,1,Gradient Boosting,88.6076,0.9890,0.8404,0.9806,0.9086
1,2,Random Forest,88.9241,0.9944,0.8404,0.9903,0.9109
2,3,Ridge Regression,85.4430,0.9771,0.8028,0.9612,0.8814
3,4,Linear Regression,86.0759,0.9774,0.8122,0.9612,0.8872


Compare confusion matrix values

In [13]:
confusion_columns = [
    "Model",
    "TN",
    "FP",
    "FN",
    "TP"
]

confusion_comparison = results_df[
    confusion_columns
]

print("=" * 70)
print("CONFUSION MATRIX COMPARISON")
print("=" * 70)

display(confusion_comparison)

CONFUSION MATRIX COMPARISON


,Model,TN,FP,FN,TP
0,Gradient Boosting,101,2,34,179
1,Random Forest,102,1,34,179
2,Ridge Regression,99,4,42,171
3,Linear Regression,99,4,40,173


Compare overfitting and underfitting

In [14]:
baseline = DummyRegressor(
    strategy="mean"
)

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    train_rmse = row["Train RMSE"]
    val_rmse = row["Validation RMSE"]

    gap_ratio = (
        (val_rmse - train_rmse) / train_rmse
        if train_rmse > 0 else np.inf
    )

    if val_rmse >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    return "Reasonable generalization"

results_df["Fitting Assessment"] = (
    results_df.apply(assess_fitting, axis=1)
)

results_df["Gap Ratio (%)"] = (
    results_df["RMSE Gap"]
    / results_df["Train RMSE"]
    * 100
)

overfitting_comparison = results_df[
    [
        "Model",
        "Train RMSE",
        "Validation RMSE",
        "RMSE Gap",
        "Gap Ratio (%)",
        "Fitting Assessment"
    ]
]

print(f"Baseline Validation RMSE: {baseline_rmse:.4f}")

display(overfitting_comparison.round(4))

Baseline Validation RMSE: 4.5635


,Model,Train RMSE,Validation RMSE,RMSE Gap,Gap Ratio (%),Fitting Assessment
0,Gradient Boosting,0.0410,1.4427,1.4017,3415.7956,Possible overfitting
1,Random Forest,0.7316,1.4733,0.7417,101.3787,Possible overfitting
2,Ridge Regression,1.5695,1.8322,0.2626,16.7340,Reasonable generalization
3,Linear Regression,1.5492,1.8829,0.3337,21.5406,Reasonable generalization


Show the winner for each metric

In [15]:
metric_definitions = {
    "Lowest Validation RMSE": ("Validation RMSE", "min"),
    "Lowest Validation MAE": ("Validation MAE", "min"),
    "Lowest Validation MSE": ("Validation MSE", "min"),
    "Highest Validation R2": ("Validation R2", "max"),
    "Highest Exact Grade Accuracy": (
        "Exact Grade Accuracy (%)", "max"
    ),
    "Highest Within 1 Grade Accuracy": (
        "Within 1 Grade (%)", "max"
    ),
    "Highest Pass/Fail Accuracy": (
        "Pass/Fail Accuracy (%)", "max"
    ),
    "Highest F1 Score": ("F1 Score", "max")
}

metric_winners = []

for metric_name, (column, direction) in metric_definitions.items():

    if direction == "min":
        idx = results_df[column].idxmin()
    else:
        idx = results_df[column].idxmax()

    metric_winners.append({
        "Metric": metric_name,
        "Best Model": results_df.loc[idx, "Model"],
        "Value": results_df.loc[idx, column]
    })

winners_df = pd.DataFrame(metric_winners)

print("=" * 70)
print("BEST MODEL FOR EACH PERFORMANCE METRIC")
print("=" * 70)

display(winners_df.round(4))

BEST MODEL FOR EACH PERFORMANCE METRIC


,Metric,Best Model,Value
0,Lowest Validation RMSE,Gradient Boosting,1.4427
1,Lowest Validation MAE,Random Forest,0.9324
2,Lowest Validation MSE,Gradient Boosting,2.1579
3,Highest Validation R2,Gradient Boosting,0.8950
4,Highest Exact Grade Accuracy,Random Forest,48.1013
5,Highest Within 1 Grade Accuracy,Random Forest,70.5696
6,Highest Pass/Fail Accuracy,Random Forest,88.9241
7,Highest F1 Score,Random Forest,0.9109


Select the overall best model

In [16]:
best_result = results_df.iloc[0]

best_model_name = best_result["Model"]
best_experiment = best_result["Experiment"]

print("=" * 70)
print("OVERALL BEST REGRESSION MODEL")
print("=" * 70)

print("Model:", best_model_name)
print("Experiment:", best_experiment)

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation MSE  : {best_result['Validation MSE']:.4f}")
print(f"Validation R2   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy   : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision  : {best_result['Precision']:.4f}")
print(f"Recall     : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score   : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

OVERALL BEST REGRESSION MODEL
Model: Gradient Boosting
Experiment: GB-5

REGRESSION METRICS
Validation RMSE : 1.4427
Validation MAE  : 0.9385
Validation MSE  : 2.1579
Validation R2   : 0.8950
RMSE Gap        : 1.4017

GRADE ACCURACY
Exact Accuracy  : 43.99%
Within ±1 Grade : 68.99%
Within ±2 Grades: 89.56%

PASS/FAIL CLASSIFICATION
Accuracy   : 88.61%
Precision  : 0.9890
Recall     : 0.8404
Specificity: 0.9806
F1 Score   : 0.9086

CONFUSION MATRIX
TN: 101
FP: 2
FN: 34
TP: 179

FITTING ASSESSMENT
Possible overfitting
